In [ ]:
from dotenv import load_dotenv
#load_dotenv()
import pandas as pd
import numpy as np
import os
#print(os.getenv("GROQ_API_KEY"))

In [ ]:
SYSTEM_PROMPT = """
You are an alarm audit assistant.
Your role is to evaluate whether a human-assigned alarm tag is correct based on alarm context and operator comments.
You must follow the provided tag definitions exactly.
Do not invent new tags or redefine policy.
Output must be valid JSON only.
""".strip()
USER_PROMPT_TEMPLATE = """
Tag definitions:
- Nuisance: environmental triggers (wind, rain, animals)
- Valid: real incident or confirmed issue (dispatch, verified, fault)
- FALSE: no issue found

Task:
Is the assigned tag correct?

Return JSON:
{{"tag_correctness": "Valid|Invalid"}}
""".strip()

In [ ]:
def normalize_tag(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    if x.upper() == "FALSE":
        return "FALSE"
    if x.lower() == "valid":
        return "Valid"
    if x.lower() == "nuisance":
        return "Nuisance"
    return x

def prepare_eval_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # If visible Tag exists but assigned_tag does not, use it
    if "assigned_tag" not in df.columns and "Tag" in df.columns:
        df["assigned_tag"] = df["Tag"]

    df["assigned_tag"] = df["assigned_tag"].map(normalize_tag)
    df["should_be_tag"] = df["should_be_tag"].map(normalize_tag)

    # Derive mislabeled if not already present
    if "is_mislabeled" not in df.columns:
        df["is_mislabeled"] = df["assigned_tag"] != df["should_be_tag"]

    # This is the old-style gold label for evaluation
    # Valid = human tag was correct
    # Invalid = human tag was incorrect
    df["Correct Tag"] = np.where(df["is_mislabeled"], "Invalid", "Valid")

    return df

In [ ]:
def build_prompt(row):
    return USER_PROMPT_TEMPLATE.format(
        alarm_type=row["Alarm type"],
        mode=row["Mode"],
        alarm_name=row["Alarm name"],
        alarm_point=row["Alarm Point number"],
        comment=row["Full comment"],
        tag=row["assigned_tag"],
    )

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal

class Evidence(BaseModel):
    field: str
    quote: str
    why: str

class AuditResult(BaseModel):
    tag_correctness: Literal["Valid", "Invalid"]
    confidence: float = Field(ge=0.0, le=1.0)
    evidence: List[Evidence]

In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

def run_llm_eval(df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    rows = []

    for _, row in df.iterrows():
        prompt = build_prompt(row)

        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0.0,
            max_tokens=150,
        )

        raw_text = response.choices[0].message.content

        try:
            parsed = json.loads(raw_text)
            model_tag_correctness = parsed.get("tag_correctness", "Invalid")
            model_confidence = parsed.get("confidence", 0.0)
            model_evidence = parsed.get("evidence", [])
        except Exception:
            model_tag_correctness = "Invalid"
            model_confidence = 0.0
            model_evidence = []

        rows.append({
            **row.to_dict(),
            "model_tag_correctness": model_tag_correctness,
            "model_confidence": model_confidence,
            "model_evidence": model_evidence,
            "raw_model_output": raw_text,
        })

    return pd.DataFrame(rows)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def score_results(results_df: pd.DataFrame):
    y_true = results_df["Correct Tag"]
    y_pred = results_df["model_tag_correctness"]

    # Sanity check
    assert set(y_true.unique()) <= {"Valid", "Invalid"}
    assert set(y_pred.unique()) <= {"Valid", "Invalid"}

    # Accuracy
    acc = accuracy_score(y_true, y_pred)

    # Precision, Recall, F1 for Invalid (positive class)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        pos_label="Invalid",
        average="binary",
        zero_division=0
    )

    # Confusion matrix (rows = true, cols = predicted)
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=["Valid", "Invalid"]
    )

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1       : {f1:.4f}")
    print("\nConfusion Matrix [Valid, Invalid]:")
    print(cm)

    return {
        "accuracy": acc,
        "precision_invalid": precision,
        "recall_invalid": recall,
        "f1_invalid": f1,
        "confusion_matrix": cm
    }

In [ ]:
# Load your new synthetic dataset
df = pd.read_csv("synthetic_alarm_logs_with_hidden_columns.csv")
# or
# df = pd.read_parquet("new_synthetic_logs.parquet")

eval_df = prepare_eval_df(df)

results_df = run_llm_eval(eval_df, model_name="meta-llama/llama-prompt-guard-2-22m")

metrics = score_results(results_df)